<a href="https://colab.research.google.com/github/karye/Liu-labbar/blob/main/Gymnasiet_Lab_2_Maskininlarning/Lektion_3_Bygga_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🤖 Maskininlärning – Lektion 3: Bygga och träna en AI

**Målgrupp:** Gymnasiet, 16 år, inga förkunskaper krävs  
**Tid:** ca 45 minuter  
**Mål:** Förstå vad en modell är, hur den fungerar och hur vi tränar den

---

### Upphovspersoner
Originalversion: David Bergström & Mattias Tiger, mattias.tiger@liu.se  
Gymnasieversion baserad på originalverket ovan.

### Licens
CC BY-NC-SA 4.0 – https://creativecommons.org/licenses/by-nc-sa/4.0/

---
## 🌳 Del 1 – Hur tänker en AI?

Föreställ dig att du spelar **20 frågor** – ett spel där man ställer ja/nej-frågor för att gissa ett hemligt objekt.

För att gissa blomarten kanske du ställer frågor såhär:

```
"Är kronbladet kortare än 2.5 cm?"
         │
        JA ──────────────────── NEJ
         │                       │
    🌸 Setosa           "Är kronbladet kortare än 4.7 cm?"
                                 │
                              JA ─── NEJ
                                │       │
                        🌺 Versicolor  🌻 Virginica
```

Varje fråga delar upp blommorna i grupper tills vi har ett svar!  
Det kallas ett **Beslutsträd (Decision Tree)**.

### XGBoost – en skog av beslutsträd

Vi ska använda ett bibliotek som heter **XGBoost**.  
Det är som att fråga **ett helt gäng experter** (hundratals beslutsträd) och ta ett gemensamt beslut.

> 🌲🌲🌲 Tänk på det som en omröstning: om 95 av 100 träd säger "Setosa" – då är svaret troligtvis Setosa!

XGBoost är extremt kraftfullt och används av riktiga AI-ingenjörer i stora företag.

### 💬 Reflektionsfråga 3.1

Du ska gissa om en person är yngre eller äldre än 30 år.  
Du får bara ställa ja/nej-frågor om utseendet.

**Fråga:** Vilka tre frågor skulle du ställa?  
Rita gärna ett litet beslutsträd!

*(Det finns inget rätt svar – tänk kreativt!)*

---
## ⚙️ Del 2 – Förbered data (samma som Lektion 1 och 2)

Vi upprepar kodstegen från de förra lektionerna för att ha all data redo.  
Det här är ett **vanligt mönster** i maskininlärning – man alltid börjar med att ladda och dela upp datan.

In [ ]:
# Installera xgboost om det saknas (behövs ibland i Colab)
!pip install xgboost -q
print("✅ xgboost är installerat!")

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

# Ladda datasetet
iris = load_iris(as_frame=True)
X = iris.data
y = iris.target

# Dela upp i träning (80%) och test (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Träningsdata: {len(X_train)} blommor")
print(f"Testdata:     {len(X_test)} blommor")
print("✅ Data är redo!")

---
## 🏗️ Del 3 – Skapa modellen

Nu skapar vi vår **modell (Model)**. Tänk på modellen som en **tom hjärna** –  
den vet ingenting ännu, men är redo att lära sig!

`XGBClassifier` är en **klassificerare (Classifier)** – en modell som kan placera saker i kategorier.  
*(Classify = kategorisera/sortera på engelska)*

Parametern `n_estimators` styr hur många beslutsträd som ingår i "röstningskommittén".  
Parametern `max_depth` styr hur många frågor varje träd får ställa.

Vi börjar enkelt – `n_estimators=10` och `max_depth=3`:

In [ ]:
# Skapa modellen (den tomma hjärnan)
modell = XGBClassifier(
    n_estimators=10,     # Antal beslutsträd i röstningskommittén
    max_depth=3,         # Hur djupt varje träd får växa (fler frågor)
    random_state=42,
    eval_metric='mlogloss',
    verbosity=0
)

print("✅ Modellen är skapad (men inte tränad ännu)!")
print("Den vet ingenting om blommor – ännu...")

---
## 🎓 Del 4 – Träna modellen!

Nu är det dags för det viktigaste steget – **träningen (Training)**!  
Vi "matar" modellen med träningsdatan så att den kan hitta mönster.

I Python gör vi detta med metoden **`.fit()`** – som i "fit" (anpassa på engelska).  
Vi anpassar modellen till datan.

Vi ger den:
- `X_train` – de 120 blommornas mätningar (framsidan på flashcards)
- `y_train` – de rätta arterna för dessa 120 blommor (baksidan på flashcards)

Kör cellen nedan och se hur snabbt det går:

In [ ]:
# Träna modellen på träningsdatan!
modell.fit(X_train, y_train)

print("✅ Träningen är klar!")
print(f"Modellen har nu lärt sig av {len(X_train)} blommor.")

### Wow, det gick snabbt!

Datorn hann gå igenom alla 120 blommor och hitta mönster på bråkdelen av en sekund!  
Jämför det med hur lång tid det tar för en människa att memorera 120 flashcards.

> 🚀 En av superkrafterna med maskininlärning är att datorer kan lära sig från  
> **miljontals exempel** på några minuter – något som tar en människa flera liv.

---
## 🔮 Del 5 – Testa modellen (en sneak preview)

Även om vi utvärderar modellen ordentligt i Lektion 4, kan vi göra ett litet test nu!  

Låt oss låta modellen gissa arten för en blomma med dessa mått:
- kalkblad längd: 5.1 cm
- kalkblad bredd: 3.5 cm
- kronblad längd: 1.4 cm
- kronblad bredd: 0.2 cm

In [ ]:
import pandas as pd

# En enda blomma att gissa
ny_blomma = pd.DataFrame([[5.1, 3.5, 1.4, 0.2]],
                         columns=X_train.columns)

# Be modellen gissa
gissning = modell.predict(ny_blomma)[0]

artnamn = {0: 'Setosa', 1: 'Versicolor', 2: 'Virginica'}
print(f"AI:ns gissning: {artnamn[gissning]}")
print("(Rätt svar: Setosa – det är den första blomman i datasetet!")

---
## 🎯 Del 6 – Din uppgift: Experimentera med modellen

**Uppgift 1:** Gå tillbaka till cellen i **Del 3** (Skapa modellen).  
Ändra `n_estimators=10` till `n_estimators=100` och träna om modellen.  
- Tog det längre tid? (Troligtvis inte märkbart – datorer är snabba!)
- Fler träd = troligtvis bättre precision, men tar lite längre tid

**Uppgift 2:** I cellen i **Del 5**, ändra mätningarna till:  
`[[6.3, 3.3, 6.0, 2.5]]`  
Vad gissar AI:n nu? (Rätt svar: Virginica)

---
## 💡 Del 7 – Sammanfattning

| Begrepp | Förklaring | Engelskt namn |
|---------|------------|---------------|
| **Modell** | AI:ns "hjärna" som lär sig mönster | Model |
| **Klassificerare** | En modell som sorterar saker i kategorier | Classifier |
| **Beslutsträd** | Modell som ställer ja/nej-frågor | Decision Tree |
| **Träning** | Processen där modellen lär sig av data | Training / Fitting |
| **Förutsägelse** | Modellens gissning på ny data | Prediction |

### Resan hittills:

```
✅ Steg 1: Förstå data     (Lektion 1)
✅ Steg 2: Dela upp data   (Lektion 2)
✅ Steg 3: Träna modellen  (Lektion 3 – detta!)
⏳ Steg 4: Utvärdera       (Lektion 4)
⏳ Steg 5: Verklig data    (Lektion 5)
```

### 🚀 Nästa lektion

I **Lektion 4** ska vi **utvärdera** hur bra AI:n faktiskt blev!  
Vi låter den göra "provet" (testdatan) och rättar svaren.